In [8]:


using Gen
using Plots
using Statistics
using Distributions 
using LinearAlgebra



function compute_rhat(chains)
    m = length(chains)   # Number of chains
    n = length(chains[1])  # Number of samples per chain
   
    # Compute chain means
    chain_means = [mean(chain) for chain in chains]
    grand_mean = mean(chain_means)

    # Compute within-chain variance W
    W = mean([var(chain, corrected=true) for chain in chains])  # corrected=true uses n-1 in denominator
   
    # Compute between-chain variance B
    B = (n / (m - 1)) * sum((chain_mean - grand_mean)^2 for chain_mean in chain_means)
    
    # Compute potential scale reduction factor (R-hat)
    var_hat = ((n - 1) / n) * W + (B / n)
    r_hat = sqrt(var_hat / W)
    
    return r_hat
end






@gen function linear_regression_model(X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma)
    # Sample the prior for intercept alpha and regression coefficients beta
    alpha ~ normal(0, sqrt(sigma_alpha2))            # Intercept
    beta = [{(:beta, i)} ~ normal(mu_beta, sqrt(sigma_beta2)) for i in 1:K]      # Regression coefficients
    
   
    nu ~ gamma(2, 10)                               
    sigma ~ exponential(lambda_sigma)                 

    # Compute the mean for each y_i: μ_i = α + X_i * β
    for i in 1:N
        mu_i = alpha + dot(X[i, :], beta)           # Linear model: μ_i = α + X_i * β
        {(:y, i)} ~ normal(mu_i, sigma)           
    end
end

DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Any, Any, Any, Any, Any, Any, Any], false, Union{Nothing, Some{Any}}[nothing, nothing, nothing, nothing, nothing, nothing, nothing], Main.var"##linear_regression_model#231", Bool[0, 0, 0, 0, 0, 0, 0], false)

In [31]:
function get_observations(y_obs)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end
    return observations
end



#Inference Important Sampling
function do_inference_is_linear(model, X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma, y_obs, num_samples)
    args = (X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma)
    observations = get_observations(y_obs)
    
    (traces, log_weights, _) = Gen.importance_sampling(
        model,
        args,
        observations,
        num_samples
    )
    
    weights = exp.(log_weights)
    
    alpha_samples = [get_choices(t)[:alpha] for t in traces]
    beta_samples = [[get_choices(t)[(:beta, k)] for k in 1:K] for t in traces]
    nu_samples = [get_choices(t)[:nu] for t in traces]
    sigma_samples = [get_choices(t)[:sigma] for t in traces]
    
    return (alpha_samples, beta_samples, nu_samples, sigma_samples, weights)
end








#Inference MH 
function do_inference_mh_linear(model, X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma, y_obs, num_iters)
    args = (X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma)
    observations = get_observations(y_obs)
    
    # Initialize trace
    trace, = generate(model, args, observations)
    accepted = 0
    
    # Storage
    alpha_samples = []
    beta_samples = Vector{Vector{Float64}}(undef, num_iters)
    nu_samples = []
    sigma_samples = []
    
    # Create proper selection
    selection = select(:alpha, :nu, :sigma, [(:beta, k) for k in 1:K]...)
    
    for i in 1:num_iters
        # Use standard Metropolis-Hastings
        new_trace, acc = metropolis_hastings(trace, selection)
        accepted += acc
        trace = new_trace
        
        # Store samples
        choices = get_choices(trace)
        push!(alpha_samples, choices[:alpha])
        beta_samples[i] = [choices[(:beta, k)] for k in 1:K]
        push!(nu_samples, choices[:nu])
        push!(sigma_samples, choices[:sigma])
    end
    
    (alpha_samples, beta_samples, nu_samples, sigma_samples, accepted/num_iters)
end

#Inference for HMC
function do_inference_hmc_linear(model, X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma, y_obs, num_iters; L=10, eps=0.1)
    args = (X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma)
    observations = get_observations(y_obs)
    
    trace, = generate(model, args, observations)
    accepted = 0
    
    alpha_samples = []
    beta_samples = Matrix{Float64}(undef, num_iters, K)
    nu_samples = []
    sigma_samples = []
    
    # Select all continuous parameters
    selection = select(:alpha, :nu, :sigma, (:beta, k) for k in 1:K...)
    
    for i in 1:num_iters
        new_trace, acc = hmc(trace, selection; L=L, eps=eps)
        accepted += acc
        trace = new_trace
        
        # Store samples
        choices = get_choices(trace)
        push!(alpha_samples, choices[:alpha])
        beta_samples[i, :] = [choices[(:beta, k)] for k in 1:K]
        push!(nu_samples, choices[:nu])
        push!(sigma_samples, choices[:sigma])
    end
    
    (alpha_samples, beta_samples, nu_samples, sigma_samples, accepted/num_iters)
end


#Multi_variable Metropolis hasting proposal function, it functions like a multi variable metropolis hastings with custom stepsize eps for each variable
function linear_regression_proposal(trace, model, args, eps_alpha, eps_beta, eps_nu, eps_sigma)
    choices = get_choices(trace)
    
    # Propose new values 
    alpha_new = choices[:alpha] + eps_alpha * randn()
    
  
    K = args[3]
    beta_new = [choices[(:beta, i)] + eps_beta * randn() for i in 1:K] 
    
    # Log-space proposals for positive parameters
    nu_new = exp(log(choices[:nu]) + eps_nu * randn())
    sigma_new = exp(log(choices[:sigma]) + eps_sigma * randn())
    
    # Create new choice map
    new_choices = choicemap()
    new_choices[:alpha] = alpha_new
    for i in 1:K
        new_choices[(:beta, i)] = beta_new[i]
    end
    new_choices[:nu] = nu_new
    new_choices[:sigma] = sigma_new
    
    # Update trace
    new_trace, = update(trace, args, (), new_choices)
    acceptance_ratio = exp(get_score(new_trace) - get_score(trace))
    
    rand() < acceptance_ratio ? (new_trace, 1) : (trace, 0)
end

#Inference using adaptive stepsizes during a burn in period.
function do_inference_adaptive_linear(model, X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma, y_obs, num_iters; target_accept=0.3)
    args = (X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma)
    observations = get_observations(y_obs)
    
    trace, = generate(model, args, observations)
    accepted = 0
    
    # Initialize step sizes
    eps_alpha, eps_beta, eps_nu, eps_sigma = 0.05, 0.05, 0.05, 0.05
    adapt_window = 100
    adapt_factor = 1.05
    
    alpha_samples = []
    beta_samples = Matrix{Float64}(undef, num_iters, K)
    nu_samples = []
    sigma_samples = []
    
    for iter in 1:num_iters
        trace, acc = linear_regression_proposal(trace, model, args, eps_alpha, eps_beta, eps_nu, eps_sigma)
        accepted += acc
        
        # Adaptation during burn-in (first 20%)
        if iter <= 0.2*num_iters && iter % adapt_window == 0
            current_accept = accepted / adapt_window
            adjustment = current_accept < target_accept ? 0.95 : adapt_factor
            
            eps_alpha *= adjustment
            eps_beta *= adjustment
            eps_nu *= adjustment
            eps_sigma *= adjustment
            accepted = 0
        end
        
        # Store samples after burn-in
        if iter > 0.2*num_iters
            choices = get_choices(trace)
            push!(alpha_samples, choices[:alpha])
            beta_samples[iter, :] = [choices[(:beta, k)] for k in 1:K]
            push!(nu_samples, choices[:nu])
            push!(sigma_samples, choices[:sigma])
        end
    end
    
    (alpha_samples, beta_samples, nu_samples, sigma_samples, accepted/num_iters)
end





do_inference_adaptive_linear (generic function with 1 method)

In [32]:
using Gen
using Statistics

# Generate synthetic data
N = 100  # Number of data points
K = 3    # Number of features
X = randn(N, K)  # Generate random features
true_alpha = 1.5
true_beta = [2.0, -1.5, 0.5]
true_sigma = 0.8

# Generate synthetic observations
y_obs = true_alpha .+ X * true_beta .+ randn(N) * true_sigma

# Model hyperparameters
sigma_alpha2 = 1.0
mu_beta = 0.0
sigma_beta2 = 1.0
lambda_sigma = 1.0

num_chains = 20
num_samples = 2000

methods = [
    ("Importance Sampling", :is),
    ("Gen Metropolis", :mh),
    ("Custom Metropolis", :custom_mh),
    ("Hamiltonian MC", :hmc)
]


for (method_name, method) in methods
    println("\n=== $method_name ===")
    
    chains_alpha = []
    chains_nu = []
    chains_sigma = []
    acc_rates = []
    
    for _ in 1:num_chains
        if method == :is
            alpha, _, nu, sigma, _ = do_inference_is_linear(
                linear_regression_model,
                X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma,
                y_obs, num_samples
            )
            push!(chains_alpha, alpha)
            push!(chains_nu, nu)
            push!(chains_sigma, sigma)
            
        elseif method == :mh
            alpha, _, nu, sigma, acc = do_inference_mh_linear(
                linear_regression_model,
                X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma,
                y_obs, num_samples
            )
            push!(chains_alpha, alpha)
            push!(chains_nu, nu)
            push!(chains_sigma, sigma)
            push!(acc_rates, acc)
            
        elseif method == :custom_mh
            
            total_iters = Int(ceil(num_samples / 0.8))
            
            alpha, _, nu, sigma, acc = do_inference_adaptive_linear(
                linear_regression_model,
                X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma,
                y_obs, total_iters 
            )
            
            push!(chains_alpha, alpha)
            push!(chains_nu, nu)
            push!(chains_sigma, sigma)
            push!(acc_rates, acc)
            
        else  # HMC
            alpha, _, nu, sigma, acc = do_inference_hmc_linear(
                linear_regression_model,
                X, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma,
                y_obs, num_samples
            )
            push!(chains_alpha, alpha)
            push!(chains_nu, nu)
            push!(chains_sigma, sigma)
            push!(acc_rates, acc)
        end
    end

    # Results
    if method != :is
        println("α R̂: ", compute_rhat(chains_alpha))
        println("ν R̂: ", compute_rhat(chains_nu))
        println("σ R̂: ", compute_rhat(chains_sigma))
        println("Acceptance rate: ", mean(acc_rates))
    end
    
    println("α mean: ", mean(vcat(chains_alpha...)))
    println("ν mean: ", mean(vcat(chains_nu...)))
    println("σ mean: ", mean(vcat(chains_sigma...)))
end


=== Importance Sampling ===
α mean: -0.006158435747809511
ν mean: 20.008313994430324
σ mean: 0.9941042111370121

=== Gen Metropolis ===
1.222087323405185
ν R̂: 1.3056292166105572
σ R̂: 1.1901312250292453
Acceptance rate: 0.0037750000000000014
α mean: 1.197162863225523
ν mean: 20.21662101846624
σ mean: 1.4509961727320224

=== Custom Metropolis ===
α R̂: 1.0054416898014837
ν R̂: 1.407036943255267
σ R̂: 1.0024598008189263
Acceptance rate: 0.30565999999999993
α mean: 1.534668168377109
ν mean: 13.284773737305775
σ mean: 0.762878901657618

=== Hamiltonian MC ===
α R̂: 4.432140780978299
ν R̂: 1.8216184919069947
σ R̂: 4.524966292114577
Acceptance rate: 0.32575000000000004
α mean: 0.2509377807728389
ν mean: 20.2751844805105
σ mean: 1.4881472622655205
